In [ ]:
%sql
/* ===================================================================================
   EBA04 - Preview newly added rows in the combined table
   These are rows in ritm16070584_combined_04272026 that do NOT exist in the
   original 7 filtered files. Uses the same EXCEPT logic from eba04.
=================================================================================== */

SELECT DateExpenseIncurrred, PublicOfficial, Expensecategory, Total, BusinessPurpose, Account, Divisor
FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026
WHERE (DateExpenseIncurrred, PublicOfficial, Expensecategory, Total, BusinessPurpose, Account)
IN (
    SELECT DateExpenseIncurrred, PublicOfficial, Expensecategory, Total, BusinessPurpose, Account
    FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026
    EXCEPT
    SELECT DateExpenseIncurrred, PublicOfficial, Expensecategory, Total, BusinessPurpose, Account
    FROM (
        SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
        UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
    )
)
ORDER BY DateExpenseIncurrred, Account

In [ ]:
# ===================================================================================
# EXPORT: EBA04 - Newly added rows -> CSV
# Output path: dbfs:/tmp/eba04_new_rows.csv
# ===================================================================================

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

def export_single_csv(df, output_dir, final_path):
    safe_rm(output_dir)
    safe_rm(final_path)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_dir)
    part_file = [f.path for f in dbutils.fs.ls(output_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_path)
    safe_rm(output_dir)
    return final_path

# --- Find the new-only row signatures via EXCEPT ---
old_union_tables = [
    "hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered",
]

key_cols = ["DateExpenseIncurrred", "PublicOfficial", "Expensecategory", "Total", "BusinessPurpose", "Account"]

combined_table = "hive_metastore.ra_adido_2025.ritm16070584_combined_04272026"

df_combined = spark.table(combined_table)

# Build old union
df_old = None
for t in old_union_tables:
    df_t = spark.table(t).select(*key_cols)
    df_old = df_t if df_old is None else df_old.unionAll(df_t)

# EXCEPT to find new-only signatures
df_new_keys = df_combined.select(*key_cols).exceptAll(df_old)

# Join back to get all columns including Divisor
df_new_rows = df_combined.join(
    df_new_keys,
    on=key_cols,
    how="inner"
).orderBy("DateExpenseIncurrred", "Account")

row_count = df_new_rows.count()
print(f"New rows found: {row_count}")

output_dir = "dbfs:/tmp/eba04_new_rows_staging"
final_path = "dbfs:/tmp/eba04_new_rows.csv"

export_single_csv(df_new_rows, output_dir, final_path)
print(f"Exported to: {final_path}")
print(f"Total rows: {row_count}")